# Cleanup: CloudSQL Census

This notebook cleans up resources created by `setup/04_setup_cloudsql_census.ipynb`.

**Resources that will be deleted:**

**1. CloudSQL Instance:**
   - PostgreSQL instance: `census-demo-db`
   - Database: `census_data` (automatically deleted with instance)
   - Table: `census_residence_type` (automatically deleted with database)
   - All indexes (automatically deleted with table)

**⚠️  WARNING:**
- This operation cannot be undone
- All data will be permanently deleted
- This operation takes approximately 5-10 minutes

**Required permissions:**
- `roles/cloudsql.admin` (to delete CloudSQL instance)

## Section 1: Configuration & Authentication

In [ ]:
# Configuration variables
PROJECT_ID = "johnswain-1200-20260303091357"  
REGION = "us-central1"                           

# CloudSQL
INSTANCE_NAME = "census-demo-db"

print("📋 Configuration:")
print(f"  Project ID: {PROJECT_ID}")
print(f"  Region: {REGION}")
print(f"  CloudSQL Instance to delete: {INSTANCE_NAME}")

In [ ]:
# Verify authentication
from google.auth import default
import google.auth

try:
    credentials, project = default()
    print("🔐 Authentication Status:")
    print(f"  ✅ Authenticated successfully")
    print(f"  ✅ Default project: {project}")
    
    if project != PROJECT_ID:
        print(f"  ⚠️  Note: Using PROJECT_ID ({PROJECT_ID}) instead of default ({project})")
    
except Exception as e:
    print(f"❌ Authentication failed: {e}")
    print("Please run: gcloud auth application-default login")
    raise

In [ ]:
# Install required libraries
import sys
import subprocess

print("📦 Installing required libraries...")
packages = ["requests"]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

print("✅ Libraries installed")

In [ ]:
# Initialize clients
import requests
import time
from google.auth.transport.requests import Request

# Get credentials for REST API calls
credentials.refresh(Request())

print("✅ Clients initialized:")
print("  - Authentication credentials")

## Section 2: Check CloudSQL Instance

Check if the CloudSQL instance exists before attempting deletion.

In [ ]:
# Check if CloudSQL instance exists
print("🔍 Checking if CloudSQL instance exists...")
print()

cloudsql_url = f"https://sqladmin.googleapis.com/v1/projects/{PROJECT_ID}/instances/{INSTANCE_NAME}"
headers = {
    "Authorization": f"Bearer {credentials.token}",
    "Content-Type": "application/json"
}

try:
    response = requests.get(cloudsql_url, headers=headers)
    
    if response.status_code == 200:
        instance_info = response.json()
        print(f"✅ CloudSQL instance '{INSTANCE_NAME}' found")
        print(f"   State: {instance_info.get('state', 'UNKNOWN')}")
        print(f"   Version: {instance_info.get('databaseVersion', 'UNKNOWN')}")
        CLOUDSQL_EXISTS = True
    elif response.status_code == 404:
        print(f"⚠️  CloudSQL instance '{INSTANCE_NAME}' not found (already deleted or never created)")
        CLOUDSQL_EXISTS = False
    else:
        print(f"⚠️  Error checking instance: {response.status_code}")
        print(f"   Response: {response.text[:200]}")
        CLOUDSQL_EXISTS = False
        
except Exception as e:
    print(f"⚠️  Error checking CloudSQL instance: {e}")
    CLOUDSQL_EXISTS = False

## Section 3: Disable Dataplex Integration

Before deleting the instance, disable Dataplex integration.

In [ ]:
# Disable Dataplex integration before deleting instance
print()
print("🔗 Disabling Dataplex Universal Catalog integration...")
print()

if not CLOUDSQL_EXISTS:
    print("⚠️  Instance not found - skipping Dataplex integration disable")
else:
    try:
        patch_body = {
            "settings": {
                "enableDataplexIntegration": False
            }
        }
        
        response = requests.patch(cloudsql_url, headers=headers, json=patch_body)
        
        if response.status_code in [200, 201]:
            print("✅ Dataplex integration disabled")
            operation = response.json()
            operation_name = operation.get('name')
            
            # Wait for operation to complete (quick operation, ~30 seconds)
            print("   Waiting for operation to complete...")
            max_wait = 60
            start_time = time.time()
            
            while time.time() - start_time < max_wait:
                # Refresh credentials if needed
                credentials.refresh(Request())
                headers["Authorization"] = f"Bearer {credentials.token}"
                
                op_url = f"https://sqladmin.googleapis.com/v1/{operation_name}"
                op_response = requests.get(op_url, headers=headers)
                
                if op_response.status_code == 200:
                    op_data = op_response.json()
                    if op_data.get('status') == 'DONE':
                        print("   ✅ Operation completed")
                        break
                
                time.sleep(5)
        else:
            print(f"⚠️  Could not disable integration: {response.status_code}")
            print("   Proceeding with instance deletion anyway...")
            
    except Exception as e:
        print(f"⚠️  Error disabling Dataplex integration: {e}")
        print("   Proceeding with instance deletion anyway...")

## Section 4: Delete CloudSQL Instance

Delete the CloudSQL PostgreSQL instance.

**⚠️  WARNING:** This will permanently delete the instance and all databases.

**Note:** CloudSQL deletion takes approximately 5-10 minutes to complete.

In [ ]:
# Delete CloudSQL instance
print()
print("🗑️  Deleting CloudSQL PostgreSQL instance...")
print()

if not CLOUDSQL_EXISTS:
    print("⚠️  Instance not found - skipping deletion")
else:
    print(f"⚠️  About to delete instance: {INSTANCE_NAME}")
    print("⏱️  This operation takes approximately 5-10 minutes")
    print()
    
    delete_url = f"https://sqladmin.googleapis.com/v1/projects/{PROJECT_ID}/instances/{INSTANCE_NAME}"
    
    try:
        response = requests.delete(delete_url, headers=headers)
        
        if response.status_code in [200, 202]:
            operation = response.json()
            operation_name = operation.get('name')
            print(f"✅ Instance deletion started")
            print(f"   Operation: {operation_name}")
            print()
            print("⏳ Waiting for deletion to complete...")
            
            max_wait_time = 900  # 15 minutes
            start_time = time.time()
            last_status = None
            
            while time.time() - start_time < max_wait_time:
                # Refresh credentials if needed
                credentials.refresh(Request())
                headers["Authorization"] = f"Bearer {credentials.token}"
                
                op_url = f"https://sqladmin.googleapis.com/v1/{operation_name}"
                op_response = requests.get(op_url, headers=headers)
                
                if op_response.status_code == 200:
                    op_data = op_response.json()
                    status = op_data.get('status')
                    
                    if status == 'DONE':
                        if 'error' in op_data:
                            print(f"\n❌ Instance deletion failed: {op_data['error']}")
                            break
                        else:
                            elapsed = int(time.time() - start_time)
                            minutes = elapsed // 60
                            seconds = elapsed % 60
                            print(f"\n✅ CloudSQL instance deleted successfully! (took {minutes}m {seconds}s)")
                            break
                    else:
                        elapsed = int(time.time() - start_time)
                        minutes = elapsed // 60
                        seconds = elapsed % 60
                        if status != last_status:
                            print(f"\n   Status: {status} - Elapsed: {minutes}m {seconds}s")
                            last_status = status
                        else:
                            print(f"   Status: {status} - Elapsed: {minutes}m {seconds}s", end='\r')
                
                time.sleep(15)  # Check every 15 seconds
            
            print()
            print("=" * 60)
            print("✅ CloudSQL instance deletion completed!")
            print("=" * 60)
            print(f"   Instance: {INSTANCE_NAME}")
            
        elif response.status_code == 404:
            print("⚠️  Instance not found (already deleted)")
        else:
            print(f"❌ Failed to delete instance: {response.status_code}")
            print(f"   Response: {response.text[:200]}")
            
    except Exception as e:
        print(f"❌ Error deleting CloudSQL instance: {e}")

## Section 5: Cleanup Summary

In [ ]:
# Summary
print()
print("=" * 70)
print("🎉 CLEANUP COMPLETED!")
print("=" * 70)
print()
print("📊 Summary of deleted resources:")
print()
print("   1. CloudSQL Instance:")
print(f"      - Instance: {INSTANCE_NAME}")
print("      - Database: census_data")
print("      - Table: census_residence_type")
print("      - All indexes and data")
print()
print("✅ All resources from setup_cloudsql_census.ipynb have been cleaned up!")
print()
print("🔗 Verify in Console:")
print(f"   CloudSQL Instances: https://console.cloud.google.com/sql/instances?project={PROJECT_ID}")